In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [ ]:
data = pd.read_csv(r"C:\Users\dell pc\Downloads\Student Social Media And Mental Health Impact.csv")
data

In [ ]:
# look for five rows
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
sns.histplot(data['Mental_Health_Score'],kde=True)

In [ ]:
data

In [ ]:
sns.heatmap(data.corr(numeric_only=True),annot=True)

In [ ]:
data['Stress_Level'].unique()

In [ ]:
order = ['Medium', 'Low', 'Very High', 'High']
sns.boxplot(x='Stress_Level',y='Mental_Health_Score',data=data,order=order)

In [ ]:
sns.scatterplot(x=data['Avg_Daily_Usage_Hours'],y=data['Mental_Health_Score'],data=data)

In [ ]:
sns.scatterplot(x=data['Sleep_Hours_Per_Night'],y=data['Mental_Health_Score'],data=data)

In [ ]:
platforms = data['Most_Used_Platform'].value_counts()

In [ ]:
platforms

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x=data['Most_Used_Platform'],order=data['Most_Used_Platform'].value_counts().index)

In [ ]:
num_features = data.select_dtypes(include='number')

In [ ]:
num_features

In [ ]:
#q1
#q3
#IQR
q1 = num_features.quantile(0.25)
q3 = num_features.quantile(0.75)
IQR = q3 - q1

In [ ]:
IQR

In [ ]:
lower_bound = q1 - 1.5*IQR
upper_bound = q3 + 1.5*IQR

In [ ]:
lower_bound

In [ ]:
upper_bound

In [ ]:
outlier = (num_features < lower_bound) | (num_features > upper_bound)
print(outlier.sum())

In [ ]:
# To drop duplicates 
data.drop_duplicates()

In [ ]:
data.describe()

In [ ]:
# converting negative or unrealistic value to realistic value
data['Physical_Activity_Hours'] = data['Physical_Activity_Hours'].clip(lower=0)

In [ ]:
data.describe()

In [ ]:
num_cols = data.select_dtypes(include='number')

In [ ]:
num_cols.skew()
# near to zero                : centralized 
# negative                    : left skewed
# positive(greater than zero) : Right skewed

In [ ]:
top_countries = data['Country'].value_counts().index[:10].tolist()

In [ ]:
def group_country(country):
    if country in top_countries:
        return country
    else:
        return 'other'


In [ ]:
data['Grouped_Country'] = data['Country'].apply(group_country)

In [ ]:
data['Grouped_Country']

In [ ]:
data.select_dtypes(include='number')

In [ ]:
from sklearn.model_selection import train_test_split
skews_cols = ['Study_Hours']
other_numeric_cols = ['Age',"Avg_Daily_Usage_Hours","Daily_Unlocks","Physical_Activity_Hours","Sleep_Hours_Per_Night"]

ordinal_col = ['Stress_Level']
normal_col = ["Gender","Academic_Level","Most_Used_Platform","Purpose_Of_Use","Grouped_Country"]

Feature_col = skews_cols + other_numeric_cols + ordinal_col + normal_col
x = data[Feature_col]
y = data['Mental_Health_Score']
X_train,Xtest,Y_train,Y_test = train_test_split(x,y,test_size=0.30,random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer ,StandardScaler ,OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
#1 skewed features 
skew_pipeline = Pipeline(steps=[
    ('log_transfrom',FunctionTransformer(np.log1p)),
    ('scale',StandardScaler())
])

#2 Numeric_Features 
plain_numeric_pipeline = Pipeline([
    ('scale',StandardScaler())
])

#3 Ordinal 
ordinal_pipeline = Pipeline(steps=[
    ('encode',OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))
])

#4 Nominal_Features
nominal_pipeline = Pipeline(steps=[
    ('encode',OneHotEncoder(handle_unknown='ignore'))
])
# (konsipipeline,konsafeature)
preprocessor = ColumnTransformer(transformers=[
    ('Skewed_Pipeline',skew_pipeline,skews_cols),
    ('Plain_Numeric',plain_numeric_pipeline,other_numeric_cols),
    ('Ordinal',ordinal_pipeline,ordinal_col),
    ('Normal',nominal_pipeline,normal_col)
])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score , mean_absolute_error , mean_squared_error
lr_pipeline = Pipeline(steps=[
    ('Preprocessor',preprocessor),
    ('regressor',LinearRegression())
])
lr_pipeline.fit(X_train,Y_train)
lr_preds = lr_pipeline.predict(Xtest)
lr_preds_train = lr_pipeline.predict(X_train)
lr_mae  = mean_absolute_error(Y_test,lr_preds)

lr_r2_testing = r2_score(Y_test,lr_preds)
lr_2_training = r2_score(Y_train,lr_preds_train)


print(f"Accuracy of Training :{lr_2_training} ")
print(f"Accuracy of Testing : {lr_r2_testing}")
print(f"mean_absolute_error : {lr_mae}")


In [148]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor
# Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('random forest',RandomForestRegressor(random_state=42))
])
rf_pipeline.fit(X_train,Y_train)
rf_preds = rf_pipeline.predict(Xtest)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing = r2_score(Y_test,rf_preds)
rf_r2_training = r2_score(Y_train,rf_preds_training)
rf_mae = mean_absolute_error(Y_test,rf_preds)

print(f"Accuracy of Training : {rf_r2_taining}")
print(f"Accuracy of Testing : {rf_r2_testing}")
print(f"MAE : {rf_mae}")

Accuracy of Training : 0.9827403842042827
Accuracy of Testing : 0.8917646185604018
MAE : 0.3219969777777779


In [149]:

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'random forest__n_estimators': [100, 200, 300],
    'random forest__max_depth': [5, 10, 15],
    'random forest__min_samples_split': [2, 5, 10],
    'random forest__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, Y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV R2 Score:", random_search.best_score_)
print("Test R2 Score:", random_search.score(Xtest, Y_test))


Best Parameters: {'random forest__n_estimators': 200, 'random forest__min_samples_split': 5, 'random forest__min_samples_leaf': 2, 'random forest__max_depth': 15}
Best CV R2 Score: 0.8537617055738913
Test R2 Score: 0.8795456375482766


In [155]:
from sklearn.metrics import mean_squared_error
#Calculate RMSE for LinearRegression
lr_rmse = np.sqrt(mean_squared_error(Y_test,lr_preds))
#Calculate RMSE for default Forest

rf_rmse                  = np.sqrt(mean_squared_error(Y_test,rf_preds))

#Calculate RMSE for default Random_Forest 
rf_tuned_preds           = random_search.best_estimator_.predict(Xtest)
rf_tuned_rmse            = np.sqrt(mean_squared_error(Y_test, rf_tuned_preds))
rf_tuned_training_preds  = random_search.best_estimator_.predict(X_train)
rf_tuned_training        = r2_score(Y_train, rf_tuned_training_preds)
rf_tuned_mae             = mean_absolute_error(Y_test,rf_tuned_preds)
rf_tuned_r2             = r2_score(Y_test,rf_tuned_preds)

# Create Dataframe to consolidate result 

result = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random_Forest(default)',
        'Random Forest(tuned)'
    ],
    'R2': [
        lr_r2_testing,
        rf_r2_testing,
        rf_tuned_r2
    ],
    'Training R2': [
        lr_2_training,
        rf_r2_training,
        rf_tuned_training
    ],
    'MAE': [
        lr_mae,
        rf_mae,
        rf_tuned_mae
    ],
    'RMSE': [
        lr_rmse,
        rf_rmse,
        rf_tuned_rmse
    ]
})

result



,Model,R2,Training R2,MAE,RMSE
0,Linear Regression,0.759733,0.725422,0.508590,0.644410
1,Random_Forest(default),0.891765,0.982740,0.321997,0.432514
2,Random Forest(tuned),0.879546,0.958940,0.343051,0.456275


In [160]:
# Save the model 
import joblib 
joblib.dump(rf_pipeline,'Mental_Health_Model.pkl')

['Mental_Health_Model.pkl']